In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Hospital").getOrCreate()

In [2]:
patients_csv = """patient_id,name,city,age,gender,registration_date
1,Aarav,Hyderabad,29,Male,2023-01-10
2,Priya,Bangalore,34,Female,2023-02-12
3,Rahul,Mumbai,41,Male,2023-03-14
4,Sneha,Delhi,26,Female,2023-04-15
5,Kiran,Chennai,37,Male,2023-05-11
6,Meera,Hyderabad,31,Female,2023-06-10
7,Amit,Pune,45,Male,2023-06-22
8,Neha,Delhi,28,Female,2023-07-10
9,Divya,Bangalore,33,Female,2023-07-15
10,Vikram,Mumbai,52,Male,2023-08-01
11,Farhan,Hyderabad,39,Male,2023-08-10
12,Simran,Delhi,25,Female,2023-08-21
"""
doctors_csv = """doctor_id,doctor_name,specialization,city,consultation_fee
101,Dr Sharma,Cardiology,Hyderabad,1200
102,Dr Iyer,Dermatology,Bangalore,800
103,Dr Khan,Orthopedics,Mumbai,1500
104,Dr Reddy,Pediatrics,Delhi,900
105,Dr Mehta,Neurology,Hyderabad,2000
106,Dr Nair,Cardiology,Chennai,1300
107,Dr Verma,Dermatology,Pune,850
108,Dr Rao,Orthopedics,Delhi,1400
"""

appointments_csv = """appointment_id,patient_id,doctor_id,appointment_date,status
1,1,101,2024-03-01,Completed
2,2,102,2024-03-01,Completed
3,3,103,2024-03-02,Completed
4,4,104,2024-03-02,Pending
5,5,106,2024-03-03,Completed
6,6,105,2024-03-03,Completed
7,7,107,2024-03-04,Cancelled
8,8,108,2024-03-04,Completed
9,9,102,2024-03-05,Completed
10,10,103,2024-03-05,Completed
11,11,101,2024-03-06,Pending
12,12,104,2024-03-06,Completed
13,1,105,2024-03-07,Completed
14,3,108,2024-03-07,Completed
15,6,101,2024-03-08,Cancelled
16,9,106,2024-03-08,Completed
"""

bills_csv = """bill_id,appointment_id,bill_amount,payment_mode,payment_status
1,1,1200,UPI,Paid
2,2,800,Credit Card,Paid
3,3,1500,Cash,Paid
4,4,900,UPI,Pending
5,5,1300,Debit Card,Paid
6,6,2000,Credit Card,Paid
7,7,850,Cash,Cancelled
8,8,1400,UPI,Paid
9,9,800,UPI,Paid
10,10,1500,Credit Card,Paid
11,11,1200,UPI,Pending
12,12,900,Cash,Paid
13,13,2000,Credit Card,Paid
14,14,1400,UPI,Paid
15,15,1200,Cash,Cancelled
16,16,1300,Debit Card,Paid
"""

hospital_logs_txt = """Aarav login
Priya login
Rahul appointment
Sneha login
Aarav payment
Kiran appointment
Meera login
Vikram appointment
Divya payment
Farhan login
Simran appointment
Neha payment
Amit login
Rahul payment
Meera appointment
Aarav logout
Priya payment
Divya login
Vikram payment
Farhan appointment
"""

patient_profiles_json = """[
  {
    "patient_id": 1,
    "name": "Aarav",
    "contact": {"email": "aarav@mail.com", "phone": "9000011111"},
    "allergies": ["Dust", "Peanuts"]
  },
  {
    "patient_id": 2,
    "name": "Priya",
    "contact": {"email": "priya@mail.com", "phone": "9000022222"},
    "allergies": ["Pollen"]
  },
  {
    "patient_id": 3,
    "name": "Rahul",
    "contact": {"email": "rahul@mail.com", "phone": "9000033333"},
    "allergies": ["Dust", "Milk"]
  },
  {
    "patient_id": 6,
    "name": "Meera",
    "contact": {"email": "meera@mail.com", "phone": "9000066666"},
    "allergies": ["Seafood"]
  },
  {
    "patient_id": 10,
    "name": "Vikram",
    "contact": {"email": "vikram@mail.com", "phone": "9000101010"},
    "allergies": ["Pollen", "Dust"]
  }
]
"""

with open("patients.csv", "w") as f:
    f.write(patients_csv)

with open("doctors.csv", "w") as f:
    f.write(doctors_csv)

with open("appointments.csv", "w") as f:
    f.write(appointments_csv)

with open("bills.csv", "w") as f:
    f.write(bills_csv)

with open("hospital_logs.txt", "w") as f:
    f.write(hospital_logs_txt)

with open("patient_profiles.json", "w") as f:
    f.write(patient_profiles_json)

print("Hospital datasets created successfully")

Hospital datasets created successfully


In [3]:
patients = spark.read.csv("patients.csv", header=True, inferSchema=True)
doctors = spark.read.csv("doctors.csv", header=True, inferSchema=True)
appointments = spark.read.csv("appointments.csv", header=True, inferSchema=True)
bills = spark.read.csv("bills.csv", header=True, inferSchema=True)
profiles = spark.read.json("patient_profiles.json", multiLine=True)

In [4]:
patients.show()
doctors.show()
appointments.show()
bills.show()
profiles.show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+

+---------+-----------+--------------+---------+-------

In [5]:
patients.show()
doctors.show()
appointments.show()
bills.show()

patients.printSchema()
doctors.printSchema()
appointments.printSchema()

patients.count()
doctors.count()

appointments.show(5)

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+

+---------+-----------+--------------+---------+-------

In [6]:
from pyspark.sql.functions import col

patients.select("name","city","age").show()

doctors.select("doctor_name","specialization","consultation_fee").show()

patients.withColumnRenamed("name","patient_name").show()

doctors.withColumnRenamed("doctor_name","consultant_name").show()

patients.filter(col("city")=="Hyderabad").show()

patients.filter(col("gender")=="Female").show()

patients.filter(col("age")>35).show()

doctors.filter(col("city")=="Hyderabad").show()

doctors.filter(col("specialization")=="Cardiology").show()

doctors.filter(col("consultation_fee")>1000).show()

+------+---------+---+
|  name|     city|age|
+------+---------+---+
| Aarav|Hyderabad| 29|
| Priya|Bangalore| 34|
| Rahul|   Mumbai| 41|
| Sneha|    Delhi| 26|
| Kiran|  Chennai| 37|
| Meera|Hyderabad| 31|
|  Amit|     Pune| 45|
|  Neha|    Delhi| 28|
| Divya|Bangalore| 33|
|Vikram|   Mumbai| 52|
|Farhan|Hyderabad| 39|
|Simran|    Delhi| 25|
+------+---------+---+

+-----------+--------------+----------------+
|doctor_name|specialization|consultation_fee|
+-----------+--------------+----------------+
|  Dr Sharma|    Cardiology|            1200|
|    Dr Iyer|   Dermatology|             800|
|    Dr Khan|   Orthopedics|            1500|
|   Dr Reddy|    Pediatrics|             900|
|   Dr Mehta|     Neurology|            2000|
|    Dr Nair|    Cardiology|            1300|
|   Dr Verma|   Dermatology|             850|
|     Dr Rao|   Orthopedics|            1400|
+-----------+--------------+----------------+

+----------+------------+---------+---+------+-----------------+
|patient_id|p

In [7]:
patients.orderBy("age").show()

patients.orderBy(col("age").desc()).show()

patients.orderBy(col("age").desc()).show(5)

patients.orderBy("age").show(3)

doctors.orderBy(col("consultation_fee").desc()).show()

doctors.orderBy(col("consultation_fee").desc()).show(3)

doctors.orderBy("consultation_fee").show()

appointments.orderBy("appointment_date").show()

bills.orderBy(col("bill_amount").desc()).show()

bills.orderBy(col("bill_amount").desc()).show(5)

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
+----------+------+---------+---+------+-----------------+

+----------+------+---------+---+------+---------------

In [8]:
from pyspark.sql.functions import count, avg, max, min, sum

patients.groupBy("city").count().show()

patients.groupBy("gender").count().show()

doctors.groupBy("specialization").count().show()

patients.select(avg("age")).show()

patients.select(max("age")).show()

patients.select(min("age")).show()

doctors.select(avg("consultation_fee")).show()

doctors.select(max("consultation_fee")).show()

bills.select(sum("bill_amount")).show()

bills.select(avg("bill_amount")).show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+

+------+-----+
|gender|count|
+------+-----+
|Female|    6|
|  Male|    6|
+------+-----+

+--------------+-----+
|specialization|count|
+--------------+-----+
|     Neurology|    1|
|   Dermatology|    2|
|    Cardiology|    2|
|    Pediatrics|    1|
|   Orthopedics|    2|
+--------------+-----+

+--------+
|avg(age)|
+--------+
|    35.0|
+--------+

+--------+
|max(age)|
+--------+
|      52|
+--------+

+--------+
|min(age)|
+--------+
|      25|
+--------+

+---------------------+
|avg(consultation_fee)|
+---------------------+
|              1243.75|
+---------------------+

+---------------------+
|max(consultation_fee)|
+---------------------+
|                 2000|
+---------------------+

+----------------+
|sum(bill_amount)|
+----------------+
|           20250|
+----------------+

+-------------

In [9]:
patients.join(appointments,"patient_id").show()

patients.join(appointments,"patient_id") \
.select("name","city","appointment_date","status").show()

doctors.join(appointments,"doctor_id").show()

doctors.join(appointments,"doctor_id") \
.select("doctor_name","specialization","appointment_date","status").show()

appointments.join(bills,"appointment_id").show()

appointments.join(bills,"appointment_id") \
.select("appointment_id","status","bill_amount","payment_status").show()

patients.join(appointments,"patient_id") \
.join(doctors,"doctor_id").show()

patients.join(appointments,"patient_id") \
.join(doctors,"doctor_id") \
.select("name","doctor_name","specialization","appointment_date").show()

patients.join(appointments,"patient_id") \
.join(doctors,"doctor_id") \
.join(bills,"appointment_id").show()

patients.join(appointments,"patient_id") \
.join(doctors,"doctor_id") \
.join(bills,"appointment_id") \
.select("name","doctor_name","status","bill_amount","payment_mode").show()

+----------+------+---------+---+------+-----------------+--------------+---------+----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|appointment_id|doctor_id|appointment_date|   status|
+----------+------+---------+---+------+-----------------+--------------+---------+----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|             1|      101|      2024-03-01|Completed|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|             2|      102|      2024-03-01|Completed|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|             3|      103|      2024-03-02|Completed|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|             4|      104|      2024-03-02|  Pending|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|             5|      106|      2024-03-03|Completed|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|             6|      105|      2024-03-03|Completed|
|

In [10]:
from pyspark.sql.functions import lit, concat_ws, when

patients.withColumn("age_group",
    when(col("age")<30,"Young")
    .when(col("age")<40,"Adult")
    .otherwise("Senior")).show()

patients.withColumn("hospital_name",lit("BotCampus Hospital")).show()

doctors.withColumn("fee_with_tax",col("consultation_fee")*1.18).show()

bills.withColumn("bill_with_tax",col("bill_amount")*1.18).show()

bills.withColumn("bill_in_thousands",col("bill_amount")/1000).show()

patients.withColumn("country",lit("India")).show()

doctors.withColumn("doctor_label",
    concat_ws(" - ","doctor_name","specialization")).show()

patients.withColumn("patient_label",
    concat_ws(" - ","name","city")).show()

bills.withColumn("high_bill_flag",
    when(col("bill_amount")>1500,"Yes").otherwise("No")).show()

patients.withColumn("senior_patient_flag",
    when(col("age")>40,"Yes").otherwise("No")).show()

+----------+------+---------+---+------+-----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|age_group|
+----------+------+---------+---+------+-----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|    Young|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|    Adult|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|   Senior|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|    Young|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|    Adult|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|    Adult|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|   Senior|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|    Young|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|    Adult|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   Senior|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|    Adult|
|        12|Simran|    Delhi| 25|F

In [11]:
patients.withColumn("age_category",
    when(col("age")<30,"Young")
    .when(col("age")<40,"Adult")
    .otherwise("Senior")).show()

bills.withColumn("bill_category",
    when(col("bill_amount")>1500,"High")
    .when(col("bill_amount")>1000,"Medium")
    .otherwise("Low")).show()

doctors.withColumn("fee_category",
    when(col("consultation_fee")>1500,"Premium")
    .when(col("consultation_fee")>1000,"Standard")
    .otherwise("Basic")).show()

appointments.withColumn("priority",
    when(col("status")=="Pending","High")
    .when(col("status")=="Completed","Normal")
    .otherwise("Low")).show()

patients.withColumn("zone",
    when(col("city").isin("Hyderabad","Bangalore","Chennai"),"South")
    .otherwise("Metro")).show()

+----------+------+---------+---+------+-----------------+------------+
|patient_id|  name|     city|age|gender|registration_date|age_category|
+----------+------+---------+---+------+-----------------+------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|       Young|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|       Adult|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|      Senior|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|       Young|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|       Adult|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|       Adult|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|      Senior|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|       Young|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|       Adult|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|      Senior|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|      

In [12]:
from pyspark.sql.functions import to_date, year, month, current_date, datediff

patients = patients.withColumn("registration_date",to_date("registration_date"))

patients.withColumn("year",year("registration_date")).show()

patients.withColumn("month",month("registration_date")).show()

appointments = appointments.withColumn("appointment_date",to_date("appointment_date"))

appointments.withColumn("month",month("appointment_date")).show()

appointments.groupBy("appointment_date").count().show()

appointments.groupBy(month("appointment_date")).count().show()

patients.filter(col("registration_date")>"2023-06-01").show()

patients.withColumn("days_since",
    datediff(current_date(),"registration_date")).show()

appointments.withColumn("days_since",
    datediff(current_date(),"appointment_date")).show()

+----------+------+---------+---+------+-----------------+----+
|patient_id|  name|     city|age|gender|registration_date|year|
+----------+------+---------+---+------+-----------------+----+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|2023|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|2023|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|2023|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|2023|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|2023|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|2023|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|2023|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|2023|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|2023|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|2023|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|2023|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|2023|
+----------+------+---------+---+------+

In [13]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, row_number, dense_rank, sum

w = Window.partitionBy("city").orderBy(col("age").desc())
patients.withColumn("rank",rank().over(w)).show()

patients.withColumn("row_num",row_number().over(w)).show()

w2 = Window.partitionBy("specialization").orderBy(col("consultation_fee").desc())
doctors.withColumn("dense_rank",dense_rank().over(w2)).show()

doctors.withColumn("rank",rank().over(w2)).filter(col("rank")==1).show()

patients.withColumn("rank",rank().over(w)).filter(col("rank")<=2).show()

w3 = Window.orderBy(col("consultation_fee").desc())
doctors.withColumn("rank",rank().over(w3)).show()

patients.groupBy("city").count().orderBy(col("count").desc()).show()

appointments.groupBy("doctor_id").count().orderBy(col("count").desc()).show()

w4 = Window.partitionBy("payment_mode").orderBy("bill_id")
bills.withColumn("running_total",sum("bill_amount").over(w4)).show()

w5 = Window.partitionBy("doctor_id").orderBy("appointment_id")
appointments.withColumn("running_count",count("*").over(w5)).show()

+----------+------+---------+---+------+-----------------+----+
|patient_id|  name|     city|age|gender|registration_date|rank|
+----------+------+---------+---+------+-----------------+----+
|         2| Priya|Bangalore| 34|Female|       2023-02-12|   1|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|   2|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|   1|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|   1|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|   2|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|   3|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|   1|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|   2|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|   3|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   1|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|   2|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|   1|
+----------+------+---------+---+------+

In [15]:
sc = spark.sparkContext
logs = sc.textFile("hospital_logs.txt")

logs.count()

logs.map(lambda x: x.split(" ")[0]).collect()

logs.map(lambda x: x.split(" ")[1]).collect()

logs.map(lambda x: x.split(" ")[0]).distinct().collect()

logs.map(lambda x: (x.split(" ")[1],1)) \
.reduceByKey(lambda a,b: a+b).collect()

[('login', 7), ('payment', 6), ('appointment', 6), ('logout', 1)]